# Round-Trip Conversion Test

Tests all six format-conversion paths for the simple model and verifies that
forward simulation (influx_si `--noopt`, freeflux `simulate`) agrees across
formats.

## Conversion paths tested

| # | Source → Target | NA status |
|---|---|---|
| 1 | **FreeFlux → MTF** | no-NA MID (influx_si does not model NA) |
| 2 | **FreeFlux → FML** | no-NA (default) or NA-corrected via `apply_na_correction=True` |
| 3 | **FML → MTF** | depends on FML source — use no-NA FML here |
| 4 | **FML → FreeFlux** | depends on FML source |
| 5 | **MTF → MTF** (round-trip) | no-NA |
| 6 | **MTF → FML** | NA correction applied via `apply_na_correction=True` |

## NA handling
x3cflux applies ¹³C natural abundance (p=1.109%/C) internally.  
freeflux and influx_si do **not**.

- When writing FML **for x3cflux**: use `write_fluxml(..., apply_na_correction=True)`
- When writing FML **for other tools**: use the default (`apply_na_correction=False`)
- The FML in `../FluxML/simple_model.fml` currently contains **NA-corrected MID data**
  (from the last run of `convert_simple_model.ipynb`).  We therefore start from the
  freeflux source and MTF source (both no-NA) for clean comparisons.

In [1]:
from pathlib import Path
import subprocess, sys, os
import numpy as np
import pandas as pd

BASE = Path('.').resolve()
SIMPLE = BASE.parent
FREEFLUX_SRC = SIMPLE / 'freeflux'
MTF_SRC      = SIMPLE / 'MTF'
FML_SRC      = SIMPLE / 'FluxML' / 'simple_model.fml'

RT = BASE  # roundtrip_test/ output dir

INFLUX = Path('/home/te/Projects/data_model/fluxomics_data_model/.venv-MFA_Benchmarking/bin/influx_s.py')
FF_PYTHON = Path('/home/te/Projects/data_model/fluxomics_data_model/.venv-MFA_Benchmarking/bin/python')
FF_SRC    = Path('/home/te/Projects/mfa_tools/freeflux/src')

# Ground-truth flux values (from cmfa)
TRUE_FLUXES = {'R1':100, 'R2':110, 'R3':50, 'R4':20, 'R5':20, 'R6':80, 'E_out':60, 'F_out':80}
# cmfa exact MID for F[1,2,3] — no natural abundance
CMFA_EXACT  = np.array([6.34920635e-05, 8.00761905e-01, 1.98285714e-01, 8.88888889e-04])

print('influx_si available:', INFLUX.exists())
print('freeflux python:    ', FF_PYTHON.exists())
print('freeflux source:    ', FF_SRC.exists())

influx_si available: True
freeflux python:     True
freeflux source:     True


In [2]:
# Add fluxomics_data_converter to path
sys.path.insert(0, str(SIMPLE.parent.parent / 'src'))
from fluxomics_data_converter.io import (
    parse_freeflux, parse_mtf, parse_fluxml_file,
    write_freeflux, write_mtf, write_fluxml,
)

# ── Parse all three source formats ─────────────────────────────────────────
model_ff  = parse_freeflux(FREEFLUX_SRC)
model_mtf = parse_mtf(MTF_SRC / 'simple_model')
model_fml = parse_fluxml_file(FML_SRC)

def mid_from_model(m):
    """Extract F_123 MID values from a parsed model."""
    if not m.experiments or not m.experiments[0].measurement:
        return []
    data = m.experiments[0].measurement.data
    if not data:
        return []
    pts = sorted(
        [(d.weight, d.value) for d in data.data
         if 'F_123' in d.id or 'F_123' in (d.id or '')],
        key=lambda x: x[0] if x[0] is not None else 99
    )
    return [v for _, v in pts]

print('Sources parsed')
print('  freeflux MID (no-NA):', [f'{v:.4f}' for v in mid_from_model(model_ff)])
print('  MTF      MID (no-NA):', [f'{v:.4f}' for v in mid_from_model(model_mtf)])
print('  FML      MID (NA):   ', [f'{v:.4f}' for v in mid_from_model(model_fml)])
print()
print('  cmfa exact (no-NA):  ', [f'{v:.4f}' for v in CMFA_EXACT])

Sources parsed
  freeflux MID (no-NA): ['0.0001', '0.8008', '0.1983', '0.0009']
  MTF      MID (no-NA): ['0.0001', '0.8008', '0.1983', '0.0009']
  FML      MID (NA):    ['0.0001', '0.7839', '0.2130', '0.0031']

  cmfa exact (no-NA):   ['0.0001', '0.8008', '0.1983', '0.0009']


## 1. Write all format conversions

In [3]:
# ── Conversion 1: FreeFlux → MTF ───────────────────────────────────────────
MTF_FF = RT / 'MTF_from_freeflux'
MTF_FF.mkdir(exist_ok=True)
write_mtf(model_ff, MTF_FF / 'simple_model')
print('1. FreeFlux → MTF:', sorted(p.name for p in MTF_FF.glob('*')))

# ── Conversion 2a: FreeFlux → FML (no NA — for non-x3cflux use) ────────────
FML_FF_nona = RT / 'FML_from_freeflux_nona.fml'
write_fluxml(model_ff, FML_FF_nona, apply_na_correction=False)
print('2a. FreeFlux → FML (no-NA): written')

# ── Conversion 2b: FreeFlux → FML (with NA — for x3cflux) ─────────────────
FML_FF_na = RT / 'FML_from_freeflux_na.fml'
write_fluxml(model_ff, FML_FF_na, apply_na_correction=True)
print('2b. FreeFlux → FML (NA-corrected): written')

# ── Conversion 3: FML → MTF (using no-NA FML parsed fresh from FF source) ──
# We parse the no-NA FML we just wrote to get a clean model without NA
model_fml_nona = parse_fluxml_file(FML_FF_nona)
MTF_FML = RT / 'MTF_from_fml'
MTF_FML.mkdir(exist_ok=True)
write_mtf(model_fml_nona, MTF_FML / 'simple_model')
print('3. FML(no-NA) → MTF:', sorted(p.name for p in MTF_FML.glob('*')))

# ── Conversion 4: FML → FreeFlux (no-NA FML) ───────────────────────────────
FF_FML = RT / 'freeflux_from_fml'
FF_FML.mkdir(exist_ok=True)
write_freeflux(model_fml_nona, FF_FML)
print('4. FML(no-NA) → FreeFlux:', sorted(p.name for p in FF_FML.glob('*')))

# ── Conversion 5: MTF → MTF (round-trip) ───────────────────────────────────
MTF_MTF = RT / 'MTF_from_mtf'
MTF_MTF.mkdir(exist_ok=True)
write_mtf(model_mtf, MTF_MTF / 'simple_model')
print('5. MTF → MTF (round-trip):', sorted(p.name for p in MTF_MTF.glob('*')))

# ── Conversion 6: MTF → FML (with NA — for x3cflux) ──────────────────────
FML_MTF_na = RT / 'FML_from_mtf_na.fml'
write_fluxml(model_mtf, FML_MTF_na, apply_na_correction=True)
print('6. MTF → FML (NA-corrected): written')

1. FreeFlux → MTF: ['simple_model.cnstr', 'simple_model.linp', 'simple_model.mflux', 'simple_model.miso', 'simple_model.netw', 'simple_model.opt', 'simple_model.tvar']
2a. FreeFlux → FML (no-NA): written


2b. FreeFlux → FML (NA-corrected): written
3. FML(no-NA) → MTF: ['simple_model.cnstr', 'simple_model.linp', 'simple_model.mflux', 'simple_model.miso', 'simple_model.netw', 'simple_model.opt', 'simple_model.tvar']
4. FML(no-NA) → FreeFlux: ['fluxes.tsv', 'measured_MDVs.tsv', 'measured_fluxes.tsv', 'reactions.tsv']
5. MTF → MTF (round-trip): ['simple_model.cnstr', 'simple_model.linp', 'simple_model.mflux', 'simple_model.miso', 'simple_model.netw', 'simple_model.opt', 'simple_model.tvar']
6. MTF → FML (NA-corrected): written


## 2. Forward simulation with influx_si (`--noopt`)

In [4]:
import shutil

def run_influx_noopt(mtf_dir: Path, prefix: str = 'simple_model') -> dict:
    """Run influx_si --noopt and return fluxes dict and MID list."""
    # Remove stale result dir so we get a fresh run
    res_dir = mtf_dir / f'{prefix}_res'
    if res_dir.exists():
        shutil.rmtree(res_dir)

    r = subprocess.run(
        [str(INFLUX), '--noopt', '--clownr', '1e-5', '--prefix', prefix],
        cwd=mtf_dir, capture_output=True, text=True
    )
    result = {'ok': True, 'fluxes': {}, 'mid': []}

    err_path = res_dir / f'{prefix}.err'
    if err_path.exists() and err_path.read_text().strip():
        result['ok'] = False
        result['error'] = err_path.read_text().strip()[:300]
        return result

    tvar_sim = res_dir / f'{prefix}.tvar.sim'
    if tvar_sim.exists():
        df = pd.read_csv(tvar_sim, sep='\t', comment='#')
        df = df[df['Kind'] == 'NET']
        result['fluxes'] = dict(zip(df['Name'], df['Value']))

    # influx_si --noopt writes simulated MIDs to {prefix}_res/{prefix}.miso.sim
    miso_sim = res_dir / f'{prefix}.miso.sim'
    if miso_sim.exists():
        try:
            mdf = pd.read_csv(miso_sim, sep='\t', comment='#')
            # Column 'Specie' holds the metabolite name
            f123 = mdf[mdf['Specie'] == 'F'].sort_values('Isospecies')
            result['mid'] = f123['Value'].tolist()
        except Exception as e:
            result['mid_err'] = str(e)

    return result

mtf_cases = [
    ('FF→MTF',  MTF_FF,  'FreeFlux → MTF'),
    ('FML→MTF', MTF_FML, 'FML(no-NA) → MTF'),
    ('MTF→MTF', MTF_MTF, 'MTF → MTF round-trip'),
    ('MTF-src', MTF_SRC, 'Original MTF source'),
]

results = {}
for label, mtf_dir, desc in mtf_cases:
    res = run_influx_noopt(mtf_dir)
    results[label] = res
    if res['ok']:
        core = {k: round(v, 1) for k, v in res['fluxes'].items() if k in TRUE_FLUXES}
        print(f'[{label}] {desc}: OK  fluxes={core}')
    else:
        print(f'[{label}] {desc}: FAIL — {res.get("error", "?")[:80]}')

[FF→MTF] FreeFlux → MTF: OK  fluxes={'E_out': 60, 'F_out': 80, 'R1': 100, 'R2': 110, 'R3': 50, 'R4': 20, 'R5': 20, 'R6': 80}


[FML→MTF] FML(no-NA) → MTF: OK  fluxes={'E_out': 60, 'F_out': 80, 'R1': 100, 'R2': 110, 'R3': 50, 'R4': 20, 'R5': 20, 'R6': 80}


[MTF→MTF] MTF → MTF round-trip: OK  fluxes={'E_out': 60, 'F_out': 80, 'R1': 100, 'R2': 110, 'R3': 50, 'R4': 20, 'R5': 20, 'R6': 80}


[MTF-src] Original MTF source: OK  fluxes={'E_out': 60, 'F_out': 80, 'R1': 100, 'R2': 110, 'R3': 50, 'R4': 20, 'R5': 20, 'R6': 80}


In [5]:
# ── Compare simulated MIDs from all MTF conversions ─────────────────────────
print('Forward-simulated F[1,2,3] MID comparison')
print('=' * 70)
print(f'{"Source":<18} {"M+0":>10} {"M+1":>10} {"M+2":>10} {"M+3":>10}')
print('-' * 70)
print(f'{"cmfa exact":18} {CMFA_EXACT[0]:>10.5f} {CMFA_EXACT[1]:>10.5f} {CMFA_EXACT[2]:>10.5f} {CMFA_EXACT[3]:>10.5f}')
print()
for label, res in results.items():
    if res['ok'] and res['mid']:
        m = res['mid']
        delta2 = abs(m[2] - CMFA_EXACT[2]) if len(m) > 2 else float('nan')
        print(f'{label:<18} {m[0]:>10.5f} {m[1]:>10.5f} {m[2]:>10.5f} {m[3]:>10.5f}  Δ(M+2)={delta2:.6f}')
    elif not res['ok']:
        print(f'{label:<18} FAILED')
    else:
        print(f'{label:<18} MID not available')

Forward-simulated F[1,2,3] MID comparison
Source                    M+0        M+1        M+2        M+3
----------------------------------------------------------------------
cmfa exact            0.00006    0.80076    0.19829    0.00089

FF→MTF                0.00006    0.80076    0.19829    0.00089  Δ(M+2)=0.000000
FML→MTF               0.00006    0.80076    0.19829    0.00089  Δ(M+2)=0.000000
MTF→MTF               0.00006    0.80076    0.19829    0.00089  Δ(M+2)=0.000000
MTF-src               0.00006    0.80076    0.19829    0.00089  Δ(M+2)=0.000000


In [6]:
# ── Compare estimated fluxes from all MTF forward simulations ───────────────
print('Starting flux values used (from .tvar files):')
print('=' * 70)
header = f'{"Flux":<10}' + ''.join(f'{k:>14}' for k in results)
print(header)
print('-' * 70)
all_fluxes = set()
for res in results.values():
    all_fluxes.update(res.get('fluxes', {}).keys())
for flux in sorted(all_fluxes):
    row = f'{flux:<10}'
    truth = TRUE_FLUXES.get(flux, '?')
    for label in results:
        v = results[label].get('fluxes', {}).get(flux, float('nan'))
        row += f'{v:>14.2f}'
    row += f'   (truth: {truth})'
    print(row)

Starting flux values used (from .tvar files):
Flux              FF→MTF       FML→MTF       MTF→MTF       MTF-src
----------------------------------------------------------------------
E_out              60.00         60.00         60.00         60.00   (truth: 60)
F_out              80.00         80.00         80.00         80.00   (truth: 80)
R1                100.00        100.00        100.00        100.00   (truth: 100)
R2                110.00        110.00        110.00        110.00   (truth: 110)
R3                 50.00         50.00         50.00         50.00   (truth: 50)
R4                 20.00         20.00         20.00         20.00   (truth: 20)
R5                 20.00         20.00         20.00         20.00   (truth: 20)
R6                 80.00         80.00         80.00         80.00   (truth: 80)


## 3. Forward simulation with freeflux

**Notes:**
- `FF-from-FML` fails because the converted FreeFlux files include `E_out`/`F_out` as
  explicit drain reactions (freeflux expects them implicit, not as network reactions).
  This is a known FML→FreeFlux limitation.
- `FF-src` converges to a **local minimum** (R3=0, R2≈80) rather than the ground truth
  (R3=50, R2=110). This is the same sensitivity to initialization as in the standalone
  freeflux notebook — not a converter issue. freeflux needs good starting values near
  the true solution to avoid this local minimum.

In [7]:
# Run freeflux forward simulation via subprocess (MFA_Benchmarking venv).
# freeflux Fitter.simulate() has an MRO bug; use solve() instead.
# Also: FF-from-FML has E_out/F_out as explicit reactions — freeflux solver
# may have trouble with them since they aren't in the original network spec;
# we test the original source and note the limitation for the converted dir.

def run_ff_forward(ff_dir, label, mdv_file='measured_MDVs.tsv'):
    script = (
        'from freeflux import Model\n'
        'import json, pathlib, numpy as np\n'
        f'ff_dir = pathlib.Path(r"{ff_dir}")\n'
        f'mdv_file = "{mdv_file}"\n'
        'try:\n'
        '    m = Model("test")\n'
        '    m.read_from_file(str(ff_dir / "reactions.tsv"))\n'
        '    fit = m.fitter("ss")\n'
        '    fit.set_labeling_strategy("A", labeling_pattern=["010"], percentage=[1], purity=[1])\n'
        '    fit.set_measured_MDVs_from_file(str(ff_dir / mdv_file))\n'
        '    fit.set_measured_fluxes_from_file(str(ff_dir / "measured_fluxes.tsv"))\n'
        '    fit.set_fluxes_from_file(str(ff_dir / "fluxes.tsv"))\n'
        '    fit.prepare(n_jobs=1)\n'
        '    # simulate() has MRO bug — Fitter._check_dependencies overrides Simulator\n'
        '    # without passing fit_measured_fluxes; use solve() as workaround\n'
        '    res = fit.solve(solver="slsqp", ini_fluxes=str(ff_dir / "fluxes.tsv"))\n'
        '    print("Optimization successful:", res.optimization_successful)\n'
        '    mid_arr = res.simulated_MDVs["F_123"]\n'
        '    # simulated_MDVs values are numpy arrays in this version\n'
        '    if hasattr(mid_arr, "MDV"):\n'
        '        mid_arr = mid_arr.MDV\n'
        '    mid = [round(float(v), 6) for v in mid_arr]\n'
        '    print("MID:", json.dumps(dict(zip(["M0","M1","M2","M3"], mid))))\n'
        '    keep = {"R1","R2","R3","R4","R5","R6"}\n'
        '    fx = {k: round(v,2) for k,v in res.opt_total_fluxes.items() if k in keep}\n'
        '    print("opt fluxes:", json.dumps(fx))\n'
        'except Exception as e:\n'
        '    import traceback; traceback.print_exc()\n'
    )
    r = subprocess.run([str(FF_PYTHON), '-c', script],
                       capture_output=True, text=True, timeout=120)
    print(f'[{label}]')
    for line in (r.stdout + r.stderr).strip().split('\n'):
        if line.strip():
            print(' ', line)

run_ff_forward(FREEFLUX_SRC, 'FF-src (original)',  mdv_file='measured_MID.tsv')
# FF-from-FML has E_out/F_out as explicit drain reactions in reactions.tsv;
# freeflux may not handle these correctly (they have no products in freeflux format)
run_ff_forward(FF_FML,       'FF-from-FML',        mdv_file='measured_MDVs.tsv')

[FF-src (original)]
  fitting [elapsed: 0:00:01]
  Optimization successful: True
  MID: {"M0": 0.0, "M1": 0.873243, "M2": 0.125502, "M3": 0.001255}
  opt fluxes: {"R1": 100.0, "R2": 80.54, "R3": 0.0, "R4": 9.73, "R5": 9.73, "R6": 90.27}


[FF-from-FML]
  fitting [elapsed: 0:00:01]
  Traceback (most recent call last):
    File "<string>", line 16, in <module>
    File "/home/te/Projects/data_model/fluxomics_data_model/.venv-MFA_Benchmarking/lib/python3.10/site-packages/freeflux/analysis/fit.py", line 564, in solve
      res = optModel.solve_flux(tol, max_iters)
    File "/home/te/Projects/data_model/fluxomics_data_model/.venv-MFA_Benchmarking/lib/python3.10/site-packages/freeflux/solver/nlpsolver.py", line 321, in solve_flux
      opt_obj, opt_u, is_success = self._solve_flux_slsqp(tol, max_iters, disp)
    File "/home/te/Projects/data_model/fluxomics_data_model/.venv-MFA_Benchmarking/lib/python3.10/site-packages/freeflux/solver/nlpsolver.py", line 224, in _solve_flux_slsqp
      res = minimize(
    File "/home/te/Projects/data_model/fluxomics_data_model/.venv-MFA_Benchmarking/lib/python3.10/site-packages/scipy/optimize/_minimize.py", line 701, in minimize
      res = _minimize_slsqp(fun, x0, args, jac, bounds,
    File 

## 4. Verify FML round-trip files (MTF→FML and FreeFlux→FML)

In [8]:
import re

def check_fml(path: Path, label: str):
    txt = path.read_text()
    # Extract datum values
    datums = re.findall(r'datum id="F_123" stddev="[^"]+" weight="(\d)">(.*?)</datum>', txt)
    mid = [float(v) for _, v in sorted(datums, key=lambda x: int(x[0]))]
    # Extract simulation variables
    fvs = dict(re.findall(r'fluxvalue flux="([^"]+)" type="net">(.*?)</fluxvalue>', txt))
    has_eout_cnstr = 'E_out' in txt.split('<textual>')[1].split('</textual>')[0] if '<textual>' in txt else False
    has_eout_sim   = 'flux="E_out"' in txt
    print(f'{label}:')
    print(f'  E_out in <constraints>: {has_eout_cnstr}')
    print(f'  E_out in <simulation>:  {has_eout_sim}')
    print(f'  Sim vars: {[(k,float(v)) for k,v in sorted(fvs.items())]}')
    print(f'  F_123 MID: {[f"{v:.5f}" for v in mid]}')
    if mid:
        delta2 = abs(mid[2] - CMFA_EXACT[2])
        expected_na = 0.21298  # x3cflux value at true fluxes
        delta2_na = abs(mid[2] - expected_na)
        print(f'  M+2 vs cmfa exact: Δ={delta2:.5f}  vs x3cflux-NA: Δ={delta2_na:.5f}')
    print()

check_fml(FML_FF_nona, 'FreeFlux→FML (no-NA)')
check_fml(FML_FF_na,   'FreeFlux→FML (NA-corrected for x3cflux)')
check_fml(FML_MTF_na,  'MTF→FML (NA-corrected for x3cflux)')
check_fml(FML_SRC,     'Original FML (from convert_simple_model.ipynb)')

FreeFlux→FML (no-NA):
  E_out in <constraints>: True
  E_out in <simulation>:  True
  Sim vars: [('E_out', 60.0), ('F_out', 80.0), ('R1', 100.0), ('R2', 110.0), ('R3', 50.0), ('R4', 20.0), ('R5', 20.0), ('R6', 80.0)]
  F_123 MID: ['0.00006', '0.80076', '0.19829', '0.00089']
  M+2 vs cmfa exact: Δ=0.00000  vs x3cflux-NA: Δ=0.01469

FreeFlux→FML (NA-corrected for x3cflux):
  E_out in <constraints>: True
  E_out in <simulation>:  True
  Sim vars: [('E_out', 60.0), ('F_out', 80.0), ('R1', 100.0), ('R2', 110.0), ('R3', 50.0), ('R4', 20.0), ('R5', 20.0), ('R6', 80.0)]
  F_123 MID: ['0.00006', '0.78310', '0.21365', '0.00319']
  M+2 vs cmfa exact: Δ=0.01536  vs x3cflux-NA: Δ=0.00067

MTF→FML (NA-corrected for x3cflux):
  E_out in <constraints>: True
  E_out in <simulation>:  True
  Sim vars: [('E_out', 60.000000000000014), ('F_out', 79.99999999999997), ('R4', 20.00000000000001), ('R5', 19.999999999999993), ('R6', 79.99999999999996)]
  F_123 MID: ['0.00006', '0.78310', '0.21365', '0.00319']
  M

## 5. Summary

In [9]:
print('CONVERSION ROUND-TRIP SUMMARY')
print('=' * 70)
print()
print('Ground truth fluxes:', TRUE_FLUXES)
print('cmfa exact MID (no-NA): M+0={:.5f} M+1={:.5f} M+2={:.5f} M+3={:.5f}'.format(*CMFA_EXACT))
print()
print('influx_si --noopt forward simulated MIDs:')
for label, res in results.items():
    if res['ok'] and res['mid']:
        m = res['mid']
        match = 'MATCH' if abs(m[2] - CMFA_EXACT[2]) < 1e-4 else f'Δ(M+2)={abs(m[2]-CMFA_EXACT[2]):.5f}'
        print(f'  [{label}]: M+2={m[2]:.5f}  → {match}')
print()
print('FML files for x3cflux (NA-corrected, M+2 should be ~0.213):')
for p, lbl in [(FML_FF_na, 'FF→FML+NA'), (FML_MTF_na, 'MTF→FML+NA'), (FML_SRC, 'Original FML')]:
    txt = p.read_text()
    datums = re.findall(r'weight="2">(.*?)</datum>', txt)
    v = float(datums[0]) if datums else float('nan')
    expected = abs(v - 0.21298) < 1e-3
    print(f'  [{lbl}]: M+2={v:.5f}  → {"OK" if expected else "MISMATCH"}')
print()
print('NA handling (via write_fluxml apply_na_correction parameter):')
print('  FreeFlux/MTF → FML for x3cflux: apply_na_correction=True  ✓ RESOLVED')
print('  FML(NA) → FreeFlux/MTF:         inverse correction NOT yet implemented')
print('  Workaround: start from no-NA source (freeflux or MTF) for clean conversions')

CONVERSION ROUND-TRIP SUMMARY

Ground truth fluxes: {'R1': 100, 'R2': 110, 'R3': 50, 'R4': 20, 'R5': 20, 'R6': 80, 'E_out': 60, 'F_out': 80}
cmfa exact MID (no-NA): M+0=0.00006 M+1=0.80076 M+2=0.19829 M+3=0.00089

influx_si --noopt forward simulated MIDs:
  [FF→MTF]: M+2=0.19829  → MATCH
  [FML→MTF]: M+2=0.19829  → MATCH
  [MTF→MTF]: M+2=0.19829  → MATCH
  [MTF-src]: M+2=0.19829  → MATCH

FML files for x3cflux (NA-corrected, M+2 should be ~0.213):
  [FF→FML+NA]: M+2=0.21365  → OK
  [MTF→FML+NA]: M+2=0.21365  → OK
  [Original FML]: M+2=0.21298  → OK

NA handling (via write_fluxml apply_na_correction parameter):
  FreeFlux/MTF → FML for x3cflux: apply_na_correction=True  ✓ RESOLVED
  FML(NA) → FreeFlux/MTF:         inverse correction NOT yet implemented
  Workaround: start from no-NA source (freeflux or MTF) for clean conversions
